# Storage Demo

This notebook illustrates the new multi-layer storage workflow that combines Redis caching, DuckDB metadata indexing, and persistent HDF5 storage. We benchmark different configurations (plain Pandas + HDF5, +SQL indexing, +Redis caching, and the full stack) to measure write speed, reload speed, and on-disk footprint.



In [ ]:
import gc
import numpy as np
import pandas as pd
from pathlib import Path
from time import perf_counter

try:
    from neural_analysis.metrics.distributions import pairwise_distribution_comparison_batch
except ImportError:
    # Fallback for environments where the notebook is executed against an older install
    # (e.g., stale site-packages or a different checkout on Windows). We import the
    # module and fetch the attribute dynamically if it exists.
    from neural_analysis.metrics import distributions as _distributions

    if not hasattr(_distributions, "pairwise_distribution_comparison_batch"):
        raise

    pairwise_distribution_comparison_batch = _distributions.pairwise_distribution_comparison_batch

from neural_analysis.utils.io import get_hdf5_result_summary
from neural_analysis.utils.storage.config import StorageConfig, set_config

rng = np.random.default_rng(42)
datasets = {
    "condition_a": rng.normal(size=(1280, 800)),
    "condition_b": rng.normal(loc=0.75, size=(1280, 800)),
    "condition_c": rng.normal(loc=-0.75, size=(1280, 800)),
    "condition_d": rng.normal(loc=0.5, size=(1280, 800)),
    #"condition_e": rng.normal(loc=-0.5, size=(1280, 800)),
    #"condition_f": rng.normal(loc=0.25, size=(1280, 800)),
    #"condition_g": rng.normal(loc=-0.25, size=(1280, 800)),
    #"condition_h": rng.normal(loc=0.75, size=(1280, 800)),
    #"condition_i": rng.normal(loc=-0.75, size=(1280, 800)),
    #"condition_j": rng.normal(loc=0.5, size=(1280, 800)),
    #"condition_k": rng.normal(loc=-0.5, size=(1280, 800)),
    
}

output_dir = Path("output/storage_benchmarks")
output_dir.mkdir(parents=True, exist_ok=True)



In [ ]:
def run_benchmark(label: str, use_cache: bool, use_sql_index: bool) -> dict[str, float | str]:
    """Run write/load benchmark for a given storage configuration."""
    comparison_name = f"storage_demo_{label}"
    save_path = output_dir / f"{label}.h5"
    meta_path = output_dir / f"{label}.duckdb"

    if save_path.exists():
        save_path.unlink()
    if meta_path.exists():
        meta_path.unlink()

    storage_cfg = StorageConfig(
        use_redis=use_cache,
        use_sql=use_sql_index,
        redis_host="localhost",
        redis_port=6379,
        sql_path=meta_path,
    )
    set_config(storage_cfg)

    kwargs = dict(
        data=datasets,
        metrics={"wasserstein": {}, "procrustes": {}},
        comparison_name=comparison_name,
        save_path=save_path,
        progress=False,
        use_cache=use_cache,
        use_sql_index=use_sql_index,
    )

    write_start = perf_counter()
    pairwise_distribution_comparison_batch(**kwargs, regenerate=True)
    write_seconds = perf_counter() - write_start

    load_start = perf_counter()
    pairwise_distribution_comparison_batch(**kwargs, regenerate=False)
    load_seconds = perf_counter() - load_start

    file_mb = save_path.stat().st_size / (1024 * 1024)
    summary = get_hdf5_result_summary(save_path)
    rows = len(summary)
    del summary
    gc.collect()

    return {
        "label": label,
        "use_cache": use_cache,
        "use_sql_index": use_sql_index,
        "write_seconds": write_seconds,
        "load_seconds": load_seconds,
        "file_mb": file_mb,
        "rows": rows,
        "artifact_path": str(save_path),
    }

bench_configs = [
    ("hdf5_only", False, False, "Pandas + HDF5"),
    ("hdf5_sql", False, True, "HDF5 + DuckDB metadata"),
    ("hdf5_redis", True, False, "HDF5 + Redis cache"),
    ("full_stack", True, True, "Redis + SQL + HDF5"),
]



In [ ]:
BENCHMARK_RUNS = 3

records = []
for label, use_cache, use_sql, description in bench_configs:
    for run_idx in range(1, BENCHMARK_RUNS + 1):
        result = run_benchmark(label, use_cache, use_sql)
        result["description"] = description
        result["run"] = run_idx
        records.append(result)

benchmark_df = pd.DataFrame(records)
agg_df = (
    benchmark_df.groupby(["label", "description", "use_cache", "use_sql_index"], as_index=False)
    .agg(
        write_seconds_mean=("write_seconds", "mean"),
        write_seconds_std=("write_seconds", "std"),
        load_seconds_mean=("load_seconds", "mean"),
        load_seconds_std=("load_seconds", "std"),
        file_mb_mean=("file_mb", "mean"),
        rows_mean=("rows", "mean"),
    )
    .sort_values("load_seconds_mean")
)

baseline_write = agg_df.loc[agg_df["label"] == "hdf5_only", "write_seconds_mean"].iloc[0]
baseline_load = agg_df.loc[agg_df["label"] == "hdf5_only", "load_seconds_mean"].iloc[0]
agg_df["write_speedup_vs_hdf5"] = baseline_write / agg_df["write_seconds_mean"]
agg_df["load_speedup_vs_hdf5"] = baseline_load / agg_df["load_seconds_mean"]

recommended_row = agg_df.iloc[0]
recommended_label = recommended_row["label"]
recommended_use_cache = bool(recommended_row["use_cache"])
recommended_use_sql = bool(recommended_row["use_sql_index"])

recommended_cfg = StorageConfig(
    use_redis=recommended_use_cache,
    use_sql=recommended_use_sql,
    redis_host="localhost",
    redis_port=6379,
    sql_path=output_dir / f"{recommended_label}.duckdb",
)
set_config(recommended_cfg)

# Materialize the recommended artifact so downstream cells can inspect it
run_benchmark(recommended_label, recommended_use_cache, recommended_use_sql)
best_path = output_dir / f"{recommended_label}.h5"

agg_df


,label,description,use_cache,use_sql_index,write_seconds_mean,write_seconds_std,load_seconds_mean,load_seconds_std,file_mb_mean,rows_mean,write_speedup_vs_hdf5,load_speedup_vs_hdf5
1,hdf5_only,Pandas + HDF5,False,False,8.981052,1.062866,0.037382,0.013231,0.447415,32.0,1.000000,1.000000
3,hdf5_sql,HDF5 + DuckDB metadata,False,True,8.390331,0.372869,0.046975,0.009068,0.447415,32.0,1.070405,0.795779
0,full_stack,Redis + SQL + HDF5,True,True,9.285176,1.023402,0.062220,0.005185,0.447415,32.0,0.967246,0.600803
2,hdf5_redis,HDF5 + Redis cache,True,False,8.362269,0.267374,0.062605,0.003597,0.447415,32.0,1.073997,0.597111


### Recommended configuration

The table above is averaged across three independent runs per storage profile. The winning profile is stored in `recommended_row` and its artifacts were regenerated automatically. You can now reuse this profile globally:

```python
recommended_cfg
```



In [ ]:
best_summary = get_hdf5_result_summary(best_path)
best_summary[["dataset_i", "dataset_j", "metric", "value"]].head()



,dataset_i,dataset_j,metric,value
0,condition_a,condition_a,procrustes,1.002380e-29
1,condition_a,condition_b,procrustes,6.089083e-01
2,condition_a,condition_c,procrustes,6.077479e-01
3,condition_a,condition_d,procrustes,6.078435e-01
4,condition_b,condition_a,procrustes,6.089083e-01


In [5]:
from neural_analysis.utils.io import get_hdf5_result_summary

summary = get_hdf5_result_summary(best_path)
summary["value"].describe()



count      32.000000
mean      250.402342
std       390.208846
min         0.000000
25%         0.455811
50%         0.608364
75%       450.727342
max      1200.317747
Name: value, dtype: float64